In [3]:
import cv2
import numpy as np
import mediapipe as mp
from scipy.spatial import distance as dist
import tkinter as tk
from PIL import Image, ImageTk
from tensorflow.keras.models import load_model
from playsound import playsound
import threading
import os

########################################
# LOAD MODEL
########################################
model = load_model("final_drowsy_model.h5", compile=False)

########################################
# MEDIAPIPE
########################################
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1
)

########################################
# LANDMARK INDEXES
########################################
LEFT_EYE = [33,160,158,133,153,144]
RIGHT_EYE = [362,385,387,263,373,380]

# MAR points (inner lips)
TOP_LIP = [81, 13, 82, 78]
BOTTOM_LIP = [308, 14, 312, 311]
LEFT_CORNER = 61
RIGHT_CORNER = 291

########################################
# GLOBAL VARIABLES
########################################
EAR_THRESHOLD = 0.25  # will be updated after calibration
EAR_FRAMES = 8
ear_counter = 0
MAR_THRESHOLD = 0.5  

wCNN = 0.5
wEAR = 0.3
wMAR = 0.2

alarm_on = False
cap = None
running = False

EAR_CALIB_FRAMES = 20
ear_values = []

########################################
# FUNCTIONS
########################################
def compute_ear(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C)

def sound_alarm():
    global alarm_on
    if not alarm_on:
        alarm_on = True
        def play():
            playsound("alarm.wav")
            global alarm_on
            alarm_on = False
        threading.Thread(target=play, daemon=True).start()

########################################
# TKINTER WINDOW
########################################
window = tk.Tk()
window.title("Driver Drowsiness Detection System")
window.geometry("1000x700")

title = tk.Label(window, text="Driver Drowsiness Detection", font=("Arial",20,"bold"))
title.pack(pady=10)

video_label = tk.Label(window)
video_label.pack()

control_frame = tk.Frame(window)
control_frame.pack(pady=20)

########################################
# REGISTER DRIVER PANEL WITH EAR CALIBRATION
########################################
def register_driver():
    global EAR_THRESHOLD
    reg_win = tk.Toplevel(window)
    reg_win.title("Driver Registration")
    reg_win.geometry("700x600")

    tk.Label(reg_win, text="Register Driver Face & Calibrate EAR", font=("Arial",16,"bold")).pack(pady=10)
    cam_label = tk.Label(reg_win)
    cam_label.pack()

    reg_cap = cv2.VideoCapture(0)
    message_label = tk.Label(reg_win,font=("Arial",12,"bold"))
    message_label.pack(pady=10)

    def update_reg():
        ret, frame = reg_cap.read()
        if ret:
            frame = cv2.flip(frame,1)
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(img)
            imgtk = ImageTk.PhotoImage(image=img)
            cam_label.imgtk = imgtk
            cam_label.configure(image=imgtk)
        cam_label.after(10, update_reg)

    def capture():
        global EAR_THRESHOLD
        ret, frame = reg_cap.read()
        if ret:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            cv2.imwrite("registered_face.jpg", gray)

            # Compute EAR from current frame
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb)
            if results.multi_face_landmarks:
                mesh_points_raw = results.multi_face_landmarks[0].landmark
                mesh_points = np.array([[int(p.x*frame.shape[1]), int(p.y*frame.shape[0])] for p in mesh_points_raw])
                left_eye = mesh_points[LEFT_EYE]
                right_eye = mesh_points[RIGHT_EYE]
                leftEAR = compute_ear(left_eye)
                rightEAR = compute_ear(right_eye)
                avg_ear = (leftEAR + rightEAR) / 2
                EAR_THRESHOLD = 0.7 * avg_ear
                message_label.config(text=f"Face Captured & EAR calibrated: {EAR_THRESHOLD:.3f}", fg="green")
            else:
                message_label.config(text="Face not detected. Try again.", fg="red")

    def exit_reg():
        reg_cap.release()
        reg_win.destroy()

    btn_frame = tk.Frame(reg_win)
    btn_frame.pack(pady=20)
    tk.Button(btn_frame, text="Capture Face", command=capture, width=15, height=2).grid(row=0,column=0,padx=20)
    tk.Button(btn_frame, text="Exit", command=exit_reg, width=15, height=2).grid(row=0,column=1,padx=20)

    update_reg()
########################################
# MONITORING FUNCTIONS
########################################
def start_monitoring():
    global cap, running
    cap = cv2.VideoCapture(0)
    running = True
    update_frame()

def stop_monitoring():
    global running, cap
    running = False
    if cap is not None:
        cap.release()

def exit_program():
    global cap
    if cap is not None:
        cap.release()
    window.destroy()

########################################
# UPDATE FRAME WITH FUSION ALGORITHM
########################################
def update_frame():
    global cap, ear_counter, running
    if not running:
        return

    ret, frame = cap.read()
    if not ret:
        window.after(10, update_frame)
        return

    frame = cv2.flip(frame,1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    S = 0
    W = 0
    yawn_drowsy = 0
    eye_drowsy = 0

    if results.multi_face_landmarks:
        mesh_points_raw = results.multi_face_landmarks[0].landmark
        mesh_points = np.array([[int(p.x*frame.shape[1]), int(p.y*frame.shape[0])] for p in mesh_points_raw])

        # CNN prediction
        x,y,w,h = cv2.boundingRect(mesh_points)
        face_img = frame[y:y+h, x:x+w]
        if face_img.size != 0:
            face_img_resized = cv2.resize(face_img,(224,224))
            face_img_resized = cv2.cvtColor(face_img_resized, cv2.COLOR_BGR2RGB)
            face_img_resized = face_img_resized / 255.0
            face_img_resized = np.expand_dims(face_img_resized, axis=0)
            pred = model.predict(face_img_resized, verbose=0)
            score_cnn = pred[0][0]  # class 0 = DROWSY
            S += wCNN * (1 - score_cnn)
            W += wCNN

        # EAR calculation
        left_eye = mesh_points[LEFT_EYE]
        right_eye = mesh_points[RIGHT_EYE]
        leftEAR = compute_ear(left_eye)
        rightEAR = compute_ear(right_eye)
        ear = (leftEAR + rightEAR)/2
        if ear < EAR_THRESHOLD:
            ear_counter += 1
            eye_drowsy = 1 if ear_counter >= EAR_FRAMES else 0
        else:
            ear_counter = 0
            eye_drowsy = 0
        if eye_drowsy:
            S += wEAR * eye_drowsy
            W += wEAR

        # MAR calculation
        def get_point(idx):
            lm = mesh_points[idx]
            return [lm[0], lm[1]]

        verticals = [dist.euclidean(get_point(t), get_point(b)) for t,b in zip(TOP_LIP, BOTTOM_LIP)]
        avg_vertical = sum(verticals)/len(verticals)
        horizontal = dist.euclidean(get_point(LEFT_CORNER), get_point(RIGHT_CORNER))
        MAR = avg_vertical / horizontal
        yawn_drowsy = 1 if MAR > MAR_THRESHOLD else 0
        if yawn_drowsy:
            S += wMAR * yawn_drowsy
            W += wMAR

        # Fusion score
        Pfinal = S / W if W != 0 else 0
        if Pfinal >= 0.3:
            label = "Drowsy"
            threading.Thread(target=sound_alarm, daemon=True).start()
        else:
            label = "Alert"

        color = (0,0,255) if label=="Drowsy" else (0,255,0)

        # Display
        cv2.putText(frame, label, (30,80), cv2.FONT_HERSHEY_SIMPLEX,1,color,2)
        if eye_drowsy:
            cv2.putText(frame,"Eyes Closed", (30,120), cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),2)
        if yawn_drowsy:
            cv2.putText(frame,"Yawning", (30,160), cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),2)

    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    imgtk = ImageTk.PhotoImage(image=img)
    video_label.imgtk = imgtk
    video_label.configure(image=imgtk)

    window.after(10, update_frame)

########################################
# BUTTONS
########################################
tk.Button(control_frame, text="Register Driver", command=register_driver, width=18, height=2).grid(row=0,column=0,padx=10)
tk.Button(control_frame, text="Start Monitoring", command=start_monitoring, width=18, height=2).grid(row=0,column=1,padx=10)
tk.Button(control_frame, text="Stop Monitoring", command=stop_monitoring, width=18, height=2).grid(row=0,column=2,padx=10)
tk.Button(control_frame, text="Exit", command=exit_program, width=18, height=2).grid(row=0,column=3,padx=10)

########################################
# RUN
########################################
window.mainloop()

AttributeError: module 'mediapipe' has no attribute 'solutions'